# Prueba breve de preentrenamiento del modelo de 50M

Entrena durante unas pocas actualizaciones sobre una porción pequeña de FineWeb-Edu, evalúa, genera texto y guarda un checkpoint local. Funciona con CUDA, Apple MPS o CPU.

In [1]:
## Save checpoints of validation for plotting

In [2]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("/workspace/notebooks")

assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert (PROJECT_ROOT / "src" / "llm_mini_lab").is_dir(), (
    "No existe /workspace/notebooks/src/llm_mini_lab"
)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    f"{PROJECT_ROOT}[training]",
    "jupyterlab",
    "ipywidgets",
])

print("Instalado correctamente desde:", PROJECT_ROOT)

Obtaining file:///workspace/notebooks
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llm-mini-lab (pyproject.toml): started
  Building editable for llm-mini-lab (pyproject.toml): finished with status 'done'
  Created wheel for llm-mini-lab: filename=llm_mini_lab-0.1.0-0.editable-py3-none-any.whl size=1458 sha256=05eaccc0b14619599bdc4982f69c3b03bc3332b0aab142f7752abab429da4a71
  Stored in directory: /tmp/pip-ephem-wheel-cache-dagvu936/wheels/86/0d/6e/72285dfc8a169923c76a5d53bb7ec7beee15543a4bca49994e
Successfully built llm

In [3]:
%pip install -q -e ".[training]" jupyterlab ipywidgets

ERROR: file:///workspace/notebooks/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import tiktoken

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'src' / 'llm_mini_lab').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto llm-mini-lab')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from llm_mini_lab.model import GPTModel
from llm_mini_lab.pretraining import (
    GPT_CONFIG_50M, create_dataloader_smollm, generate_text_simple,
    text_to_token_ids, token_ids_to_text,
)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('PyTorch:', torch.__version__)
print('Dispositivo:', device)
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

PyTorch: 2.11.0+cu128
Dispositivo: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM (GiB): 11.6


In [5]:
SEED = 123
MAX_LENGTH = 128
BATCH_SIZE = 2
MAX_TOKENS = 1_000_000
MAX_UPDATES = MAX_TOKENS // (BATCH_SIZE * MAX_LENGTH)
VAL_MOD = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1

EVAL_INTERVAL = 10
VAL_BATCHES = 10  # Número de batches usados en cada validación

torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

In [8]:
from pathlib import Path

import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class LocalTextDataset(Dataset):
    def __init__(self, file_path, max_length):
        text = Path(file_path).read_text(encoding="utf-8")

        tokenizer = tiktoken.get_encoding("gpt2")
        self.tokens = tokenizer.encode(
            text,
            allowed_special={"<|endoftext|>"}
        )

        self.max_length = max_length
    def __len__(self):
        # Bloques consecutivos no solapados
        return max(0, (len(self.tokens) - 1) // self.max_length)
    def __getitem__(self, index):
        start = index * self.max_length
        end = start + self.max_length

        inputs = torch.tensor(
            self.tokens[start:end],
            dtype=torch.long
        )
        targets = torch.tensor(
            self.tokens[start + 1:end + 1],
            dtype=torch.long
        )
        return inputs, targets
DATA_DIR = Path("data/smollm_local")

train_dataset = LocalTextDataset(
    "/workspace/notebooks/data/smollm_local/train.txt",
    max_length=MAX_LENGTH
)
val_dataset = LocalTextDataset(
    "/workspace/notebooks/data/smollm_local/validation.txt",
    max_length=MAX_LENGTH
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0
)
# La evaluación utilizará ahora exclusivamente validation.txt
val_eval_loader = val_loader

xb, yb = next(iter(train_loader))

assert xb.shape == yb.shape
assert torch.equal(yb[:, :-1], xb[:, 1:])

print("Micro-batch:", tuple(xb.shape))
print("Bloques de entrenamiento:", len(train_dataset))
print("Bloques de validación:", len(val_dataset))
print("Tokens de entrenamiento:", len(train_dataset.tokens))
print("Tokens de validación:", len(val_dataset.tokens))

Micro-batch: (2, 128)
Bloques de entrenamiento: 5235
Bloques de validación: 558
Tokens de entrenamiento: 670193
Tokens de validación: 71449


In [9]:
from pathlib import Path

print("Directorio actual:", Path.cwd())

print("\nArchivos TXT encontrados:")
for path in Path("/workspace").rglob("*.txt"):
    print(path)

Directorio actual: /workspace/notebooks/notebooks

Archivos TXT encontrados:
/workspace/.venv-backups/50280890/venv-main-2026-09-08-1520.txt
/workspace/.venv-backups/50280890/venv-main-latest.txt
/workspace/notebooks/src/llm_mini_lab.egg-info/dependency_links.txt
/workspace/notebooks/src/llm_mini_lab.egg-info/requires.txt
/workspace/notebooks/src/llm_mini_lab.egg-info/top_level.txt
/workspace/notebooks/src/llm_mini_lab.egg-info/SOURCES.txt
/workspace/notebooks/data/tinyshakespeare/input.txt
/workspace/notebooks/data/smollm_local/train.txt
/workspace/notebooks/data/smollm_local/validation.txt
/workspace/notebooks/data/smollm_local/.ipynb_checkpoints/train-checkpoint.txt


In [10]:
cfg = {**GPT_CONFIG_50M, 'context_length': MAX_LENGTH}
model = GPTModel(cfg)
model.out_head.weight = model.tok_emb.weight
model = model.to(device)

n_params = sum(parameter.numel() for parameter in model.parameters())
assert n_params <= 50_000_000, f'El modelo tiene {n_params:,} parámetros'
print(f'Parámetros entrenables: {n_params:,} ({n_params / 1e6:.2f}M)')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

Parámetros entrenables: 47,854,080 (47.85M)


In [ ]:
model.train()

train_losses = []
val_losses = []
val_steps = []
tokens_seen = 0


def evaluate(model, data_loader, max_batches=10):
    """Calcula la pérdida media sobre varios batches de validación."""
    model.eval()
    losses = []
    with torch.inference_mode():
        for batch_idx, (inputs, targets) in enumerate(data_loader):
            if batch_idx >= max_batches:
                break
            inputs = inputs.to(device)
            targets = targets.to(device)

            logits = model(inputs)
            loss = F.cross_entropy(
                logits.flatten(0, 1),
                targets.flatten()
            )
            losses.append(loss.item())
    model.train()

    if not losses:
        raise RuntimeError("El dataloader de validación no produjo ningún batch")
    return sum(losses) / len(losses)

for update, (inputs, targets) in enumerate(train_loader, start=1):
    inputs = inputs.to(device)
    targets = targets.to(device)

    optimizer.zero_grad(set_to_none=True)

    logits = model(inputs)
    loss = F.cross_entropy(
        logits.flatten(0, 1),
        targets.flatten()
    )

    if not torch.isfinite(loss):
        raise FloatingPointError(
            f"Pérdida no finita en la actualización {update}"
        )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_losses.append(loss.item())
    tokens_seen += inputs.numel()

    if update % EVAL_INTERVAL == 0:
        # Usa val_loader para una validación real.
        mean_val_loss = evaluate(
            model,
            val_loader,
            max_batches=VAL_BATCHES
        )

        val_steps.append(update)
        val_losses.append(mean_val_loss)

        print(
            f"Step {update:04d}/{MAX_UPDATES} | "
            f"train_loss {loss.item():.4f} | "
            f"val_loss {mean_val_loss:.4f}"
        )
    else:
        print(
            f"Step {update:04d}/{MAX_UPDATES} | "
            f"train_loss {loss.item():.4f}"
        )

    if update >= MAX_UPDATES:
        break

assert len(train_losses) == MAX_UPDATES, (
    "El stream terminó antes de completar la prueba"
)

print(f"Entrenamiento completado: {tokens_seen:,} tokens")

Step 0001/3906 | train_loss 330.5449
Step 0002/3906 | train_loss 281.4091
Step 0003/3906 | train_loss 200.3860
Step 0004/3906 | train_loss 131.7034
Step 0005/3906 | train_loss 96.8578
Step 0006/3906 | train_loss 69.6857
Step 0007/3906 | train_loss 73.3045
Step 0008/3906 | train_loss 71.8763
Step 0009/3906 | train_loss 67.8599
Step 0010/3906 | train_loss 66.8780 | val_loss 64.1659
Step 0011/3906 | train_loss 64.7078
Step 0012/3906 | train_loss 62.6046
Step 0013/3906 | train_loss 57.2585
Step 0014/3906 | train_loss 56.5170
Step 0015/3906 | train_loss 51.9556
Step 0016/3906 | train_loss 46.2052
Step 0017/3906 | train_loss 54.5683
Step 0018/3906 | train_loss 55.0395
Step 0019/3906 | train_loss 50.2598
Step 0020/3906 | train_loss 47.7431 | val_loss 50.4266
Step 0021/3906 | train_loss 51.7183
Step 0022/3906 | train_loss 42.3940
Step 0023/3906 | train_loss 50.6556
Step 0024/3906 | train_loss 47.8725
Step 0025/3906 | train_loss 48.3265
Step 0026/3906 | train_loss 44.9635
Step 0027/3906 | train

In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')
prompt = text_to_token_ids('Artificial intelligence', tokenizer).to(device)
generated_ids = generate_text_simple(
    model, prompt, max_new_tokens=20, context_size=MAX_LENGTH
)
print('Muestra:', token_ids_to_text(generated_ids.cpu(), tokenizer))

checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'gpt-50m-smoke-test.pt'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg,
    'updates': len(train_losses),
    'tokens_seen': tokens_seen,
}, checkpoint_path)
print('Checkpoint guardado en:', checkpoint_path)

In [ ]:
%pip install matplotlib
%pip install numpy

In [1]:
import matplotlib.pyplot as plt

train_steps = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 7))

plt.plot(
    train_steps,
    train_losses,
    label="Train loss",
    alpha=0.55
)

plt.plot(
    val_steps,
    val_losses,
    marker="o",
    linewidth=2,
    label="Validation loss (cada 10 steps)"
)

plt.title("Train y validation loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.yscale("log")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

NameError: name 'train_losses' is not defined